In [6]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [7]:
df_audio_features = pd.read_csv("../data/audio_features.csv")

## 1. Normalize numeric features per participant

In [8]:
# -------------------------
# 1. Load participant mapping
# -------------------------
df_participants = pd.read_csv("../data/pilot.csv")

df_participants = df_participants[["clue_id", "Spymaster ID"]].rename(
    columns={"Spymaster ID": "participant_id"}
)

# -------------------------
# 2. Merge
# -------------------------
df = df_audio_features.merge(df_participants, on="clue_id", how="left")

# -------------------------
# 3. Ensure clue_id matches (optional but safe)
# -------------------------
df["clue_id"] = df["clue_id"].astype(str).str.strip()
df["participant_id"] = df["participant_id"].astype(str).str.strip()

# -------------------------
# 4. Select ONLY numeric features
# -------------------------
exclude_cols = ["clue_id", "participant_id", "confidence", "difficulty"]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
features = [col for col in numeric_cols if col not in exclude_cols]

print(f"Using {len(features)} numeric features")

# -------------------------
# 5. Add deviation features
# -------------------------
for col in features:
    df[f"{col}_dev"] = df[col] - df.groupby("participant_id")[col].transform("mean")

# -------------------------
# 6. Safe z-score normalization
# -------------------------
def safe_zscore(x):
    std = x.std()
    if std == 0 or np.isnan(std):
        return x - x.mean()
    return (x - x.mean()) / std

df[features] = df.groupby("participant_id")[features].transform(safe_zscore)

# -------------------------
# 7. Fill NaNs
# -------------------------
df = df.fillna(0)

print("Done. Shape:", df.shape)

Using 58 numeric features
Done. Shape: (71, 121)


In [9]:
df.head()

,clue_id,confidence,difficulty,transcript,clue_word_frequency,duration,clue_latency,clue_number_latency,speech_rate,speech_ratio,...,mfcc_13_mean_dev,mfcc_13_std_dev,energy_mean_dev,energy_std_dev,energy_range_dev,energy_p25_dev,energy_p75_dev,jitter_dev,shimmer_dev,hnr_dev
0,1,3,2,"Okay, let me see here. So, mm-hmm. Alright, t...",0.758687,0.378222,1.483496,-0.746282,-0.106635,-0.203309,...,-0.465675,0.760346,-0.003939,0.004463,0.096480,-0.001506,-0.001961,-0.004079,0.015694,1.644406
1,2,4,1,homo sapiens to,-2.284780,-0.543093,-0.463375,0.139763,-0.720194,-1.068199,...,0.160109,0.308415,-0.008999,0.005876,-0.066027,-0.001506,-0.041481,0.004896,0.039277,-2.964881
2,3,4,2,Human 2,0.994141,-0.863681,-0.838843,1.468831,2.448223,0.441978,...,-2.438803,0.286867,0.024711,-0.024349,-0.286796,-0.001506,0.075795,0.001324,0.006339,0.182086
3,4,2,4,This is tough here. Subt x mask.ś,-0.252896,-0.146104,0.516020,-0.746282,-0.639957,-0.965797,...,1.054104,-1.948419,-0.008186,0.004418,0.028319,-0.001506,-0.039224,0.008951,0.034387,-4.907182
4,5,4,3,"See, I don't know how to link some of these. ...",-0.261616,0.629898,-0.669982,-0.398193,0.334273,1.421530,...,2.063402,0.856300,0.002208,0.001413,0.001481,0.004335,-0.010968,0.000528,-0.003851,0.088245


## 2. Map levels to 3 categories

In [10]:
def map_levels(x):
    if x in [1, 2]:
        return "low"
    elif x == 3:
        return "medium"
    elif x in [4, 5]:
        return "high"
    
df["confidence_level"] = df["confidence"].apply(map_levels)
df["difficulty_level"] = df["difficulty"].apply(map_levels)

df.drop(columns=["confidence", "difficulty", "transcript"], inplace=True)
df.head()

,clue_id,clue_word_frequency,duration,clue_latency,clue_number_latency,speech_rate,speech_ratio,articulation_rate,pause_count,pause_mean,...,energy_mean_dev,energy_std_dev,energy_range_dev,energy_p25_dev,energy_p75_dev,jitter_dev,shimmer_dev,hnr_dev,confidence_level,difficulty_level
0,1,0.758687,0.378222,1.483496,-0.746282,-0.106635,-0.203309,-0.246381,1.287504,-0.353398,...,-0.003939,0.004463,0.096480,-0.001506,-0.001961,-0.004079,0.015694,1.644406,medium,low
1,2,-2.284780,-0.543093,-0.463375,0.139763,-0.720194,-1.068199,0.050852,-0.460494,0.665544,...,-0.008999,0.005876,-0.066027,-0.001506,-0.041481,0.004896,0.039277,-2.964881,high,low
2,3,0.994141,-0.863681,-0.838843,1.468831,2.448223,0.441978,1.694925,-0.798816,-1.487240,...,0.024711,-0.024349,-0.286796,-0.001506,0.075795,0.001324,0.006339,0.182086,high,low
3,4,-0.252896,-0.146104,0.516020,-0.746282,-0.639957,-0.965797,-0.354685,0.159763,0.558579,...,-0.008186,0.004418,0.028319,-0.001506,-0.039224,0.008951,0.034387,-4.907182,low,high
4,5,-0.261616,0.629898,-0.669982,-0.398193,0.334273,1.421530,-0.528518,-0.404107,-0.518866,...,0.002208,0.001413,0.001481,0.004335,-0.010968,0.000528,-0.003851,0.088245,high,medium


In [12]:
df.to_csv("../data/clean_audio_data.csv", index=False)